# data cleaning 

this notebook prepares the raw sephora product and skincare 

goals:
- keep only fields relevant to beautyiq
- remove unnecessary columns
- handle missing values intentionally
- standardize data types
- combine the five review files
- create clean processed datasets for later sql, analysis, and dashboard use

In [1]:
import pandas as pd

In [2]:
products = pd.read_csv("../data/raw/product_info.csv")

In [3]:
review_files = [
    "../data/raw/reviews_0-250.csv",
    "../data/raw/reviews_250-500.csv",
    "../data/raw/reviews_500-750.csv",
    "../data/raw/reviews_750-1250.csv",
    "../data/raw/reviews_1250-end.csv"
]

review_dfs = []

for file in review_files:
    temp = pd.read_csv(file)
    review_dfs.append(temp)

reviews = pd.concat(review_dfs, ignore_index=True)

/var/folders/gw/b8128pz11k31ky_873j0sxsc0000gn/T/ipykernel_40032/4161635922.py:12: DtypeWarning: Columns (0: author_id) have mixed types. Specify dtype option on import or set low_memory=False.
  temp = pd.read_csv(file)


In [5]:
# what im keeping from products

product_columns = [
    "product_id",
    "product_name",
    "brand_name",
    "loves_count",
    "rating",
    "reviews",
    "price_usd",
    "limited_edition",
    "new",
    "online_only",
    "out_of_stock",
    "sephora_exclusive",
    "highlights",
    "primary_category",
    "secondary_category",
    "tertiary_category"
]

In [6]:
products_clean = products[product_columns].copy()

In [7]:
products_clean.shape

(8494, 16)

In [8]:
products_clean.head()

,product_id,product_name,brand_name,loves_count,rating,reviews,price_usd,limited_edition,new,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category
0,P473671,Fragrance Discovery Set,19-69,6320,3.6364,11.0,35.0,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Warm &Spicy Scen...",Fragrance,Value & Gift Sets,Perfume Gift Sets
1,P473668,La Habana Eau de Parfum,19-69,3827,4.1538,13.0,195.0,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume
2,P473662,Rainbow Bar Eau de Parfum,19-69,3253,4.2500,16.0,195.0,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume
3,P473660,Kasbah Eau de Parfum,19-69,3018,4.4762,21.0,195.0,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume
4,P473658,Purple Haze Eau de Parfum,19-69,2691,3.2308,13.0,195.0,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume


## clean review data

the review dataset contains over 1 million skincare reviews
this step keeps the fields needed for sentiment, customer behavior, and product opportunity analysis

In [9]:
review_columns = [
    "author_id",
    "rating",
    "is_recommended",
    "helpfulness",
    "total_feedback_count",
    "total_neg_feedback_count",
    "total_pos_feedback_count",
    "submission_time",
    "review_text",
    "review_title",
    "skin_tone",
    "eye_color",
    "skin_type",
    "hair_color",
    "product_id",
    "product_name",
    "brand_name",
    "price_usd"
]

In [10]:
reviews_clean = reviews[review_columns].copy()

In [11]:
reviews_clean.shape

(1094411, 18)

In [12]:
reviews_clean["review_text"].isna().sum()

np.int64(1444)

In [13]:
reviews_clean["submission_time"] = pd.to_datetime(
    reviews_clean["submission_time"]
)

In [14]:
reviews_clean["submission_time"].dtype

dtype('<M8[us]')

In [15]:
reviews_clean = reviews_clean.dropna(subset=["review_text"])

In [16]:
reviews_clean.shape

(1092967, 18)

In [17]:
reviews_clean.isnull().sum()

author_id                        0
rating                           0
is_recommended              167988
helpfulness                 560595
total_feedback_count             0
total_neg_feedback_count         0
total_pos_feedback_count         0
submission_time                  0
review_text                      0
review_title                309210
skin_tone                   170499
eye_color                   209566
skin_type                   111518
hair_color                  226718
product_id                       0
product_name                     0
brand_name                       0
price_usd                        0
dtype: int64

In [18]:
reviews_clean["rating"].value_counts().sort_index()

rating
1     61105
2     52956
3     81752
4    199211
5    697943
Name: count, dtype: int64

In [19]:
reviews_clean["rating"].between(1, 5).all()

np.True_

In [20]:
reviews_clean = reviews_clean.reset_index(drop=True)

In [21]:
reviews_clean.head()

,author_id,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,review_title,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd
0,1741593524,5,1.0,1.0,2,0,2,2023-02-01,I use this with the Nudestix “Citrus Clean Bal...,Taught me how to double cleanse!,NaN,brown,dry,black,P504322,Gentle Hydra-Gel Face Cleanser,NUDESTIX,19.0
1,31423088263,1,0.0,NaN,0,0,0,2023-03-21,I bought this lip mask after reading the revie...,Disappointed,NaN,NaN,NaN,NaN,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
2,5061282401,5,1.0,NaN,0,0,0,2023-03-21,My review title says it all! I get so excited ...,New Favorite Routine,light,brown,dry,blonde,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
3,6083038851,5,1.0,NaN,0,0,0,2023-03-20,I’ve always loved this formula for a long time...,Can't go wrong with any of them,NaN,brown,combination,black,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
4,47056667835,5,1.0,NaN,0,0,0,2023-03-20,"If you have dry cracked lips, this is a must h...",A must have !!!,light,hazel,combination,NaN,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0


In [22]:
products_clean["reviews"] = products_clean["reviews"].astype("Int64")

In [23]:
products_clean.dtypes

product_id                str
product_name              str
brand_name                str
loves_count             int64
rating                float64
reviews                 Int64
price_usd             float64
limited_edition         int64
new                     int64
online_only             int64
out_of_stock            int64
sephora_exclusive       int64
highlights                str
primary_category          str
secondary_category        str
tertiary_category         str
dtype: object

In [24]:
products_clean.to_csv(
    "../data/processed/products_clean.csv",
    index=False
)

reviews_clean.to_csv(
    "../data/processed/skincare_reviews_clean.csv",
    index=False
)

In [25]:
print("Products:", products_clean.shape)
print("Reviews:", reviews_clean.shape)

Products: (8494, 16)
Reviews: (1092967, 18)
